In [5]:
!pip install pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 8.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 8.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 5.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 6.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 7.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
!pip install sentence-transformers


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
!pip install ollama


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
!pip install   pandas


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [30]:
pip install openpyxl


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import chromadb
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
import os
import time
import subprocess
import ollama

/Users/aleksejvelaev/.pyenv/versions/3.12.5/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:

!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

!pkill ollama

with open("ollama_logs.txt", "w") as f:
    process = subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=f)

time.sleep(15)
!ollama list

zsh:1: command not found: apt-get
>>> Stopping running Ollama instance...
>>> Removing existing Ollama installation...
>>> Downloading Ollama for macOS...
######################################################################## 100.0%                                               0.2%                                                4.8%                                  15.0%                          23.3%                                               35.8%                         67.0%##############                      72.5%################            86.6%#####################            87.0%###################         90.7%
>>> Installing Ollama to /Applications...
>>> Starting Ollama...
>>> Install complete. You can now run 'ollama'.
]11;?\NAME               ID              SIZE      MODIFIED      
llama3.1:latest    46e0c10c039e    4.9 GB    2 minutes ago    
qwen2.5:7b         845dbda0ea48    4.7 GB    7 weeks ago      
gemma3:1b          8648f39daa8f    815 MB    7 weeks ago  

In [12]:
!ollama serve

]11;?\Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [22]:
DB_PATH = os.path.join(os.getcwd(), "chroma_db")
test_file = "/Users/aleksejvelaev/Desktop/курсовая/test_set.csv"

In [23]:
import datetime
import json 

In [24]:
cnt = 0 
UPDATE_DATE = datetime.datetime.now().strftime("%Y-%m-%d")
FILES_CONFIG = [
    {
        "file": "/Users/aleksejvelaev/Desktop/курсовая/датасет/parsser/FKN_programs_2026.jsonl",
        "category": "программы",

        "url": "https://ba.hse.ru/mirror/pubs/share/1120607478"
    },

    {
        "file": "/Users/aleksejvelaev/Desktop/курсовая/датасет/parsser/rules_fixed.jsonl",
        "category": "правила",
        "url": "https://ba.hse.ru/mirror/pubs/share/1120607430"
    },
    {

        "file": "/Users/aleksejvelaev/Desktop/курсовая/датасет/parsser/achievements_full.jsonl",
        "category": "достижения",
        "url": "https://ba.hse.ru/mirror/pubs/share/1120607604"
    }
]


model = SentenceTransformer('BAAI/bge-m3')
# print("model ok")
client = chromadb.PersistentClient(path=DB_PATH)
# print ('bd ok')
try:
    client.delete_collection(name="hse_docs_final")

except:
    # print("except")
    pass
collection = client.create_collection(name="hse_docs_final")


for config in FILES_CONFIG:
    file_path = config['file']
    with open(file_path, "r", encoding="utf-8") as f:
        # print("with")
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            content = item.get("text_for_embedding", "")
            emb = model.encode(content, normalize_embeddings=True).tolist()

            meta_data = {
                "source": file_path,
                "category": config['category'],
                "url": config['url'],
                "last_updated": UPDATE_DATE,
                "page": item.get("page", 0)
            }


            collection.add(
                ids=[f"{config['category']}_{i}"],
                embeddings=[emb],
                documents=[content],
                metadatas=[meta_data]
            )
            cnt += 1




print("ОК")
# print(cnt)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 60949.67it/s]


ОК


In [25]:
import ollama

# Проверяем наличие модели и скачиваем, если её нет
try:
    ollama.pull('qwen2.5:7b')
    print("Модель успешно загружена")
except Exception as e:
    print(f"Не удалось загрузить модель: {e}")

Модель успешно загружена


In [26]:
# Убиваем старые процессы, если они зависли
!pkill ollama
# Запускаем заново в фоновом режиме
import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'],
                     stdout=subprocess.PIPE,
                     stderr=subprocess.PIPE)
time.sleep(5) # Даем время на инициализацию
print("Сервер Ollama перезапущен")

Сервер Ollama перезапущен


In [28]:
def get_llm_score(question, reference, bot_answer):
    eval_prompt = f"""
    Ты — эксперт по оценке качества работы ИИ. 
    Оцени ответ чат-бота по сравнению с ЭТАЛОННЫМ ответом.
    
    ВОПРОС: {question}
    ЭТАЛОН: {reference}
    ОТВЕТ БОТА: {bot_answer}
    
    Оцени по критериям от 1 до 5:
    1. Точность (Accuracy): Насколько факты в ответе соответствуют эталону.
    2. Полнота (Completeness): Не упущены ли важные детали.
    
    Выдай ответ СТРОГО в формате JSON:
    {{"accuracy": оценка, "completeness": оценка, "explanation": "короткое пояснение"}}
    """
    try:
        response = ollama.generate(model='qwen2.5:7b', prompt=eval_prompt, format='json')
        import json
        return json.loads(response['response'])
    except:
        return {"accuracy": 0, "completeness": 0, "explanation": "Ошибка оценки"}

In [ ]:

import numpy as np 

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(name="hse_docs_final")
model_embed = SentenceTransformer('BAAI/bge-m3')
df_tests = pd.read_csv(test_file, sep=';')
results = []
cnt=0


hit_at_3 = []
reciprocal_ranks = []

for index, row in df_tests.iterrows():
    q = row['question']
    not_improved_q = q
    expected_ans = str(row['expected_answer']) 

    synonyms = {
    "ПМИ": "Прикладная математика и информатика",
    "ПИ": "Программная инженерия",
    "ПАД": "Прикладной анализ данных",
    "КБ": "Компьютерная безопасность",
    "ИВТ": "Информатика и вычислительная техника"
}
    processed_q = q
    for short, full in synonyms.items():
        if short in q.upper():
            processed_q = f"{q} ({full})"
            break


    
    rewrite_prompt = f"""
Ты — поисковый эксперт НИУ ВШЭ. Твоя задача: перефразировать вопрос пользователя в точный поисковый запрос для базы данных.

ПРАВИЛА:
1. ТЕРМИНОЛОГИЯ: Обязательно расшифровывай сокращения. Пиши ПОЛНОЕ название, а в скобках само сокращение.
   Список для расшифровки:
   - ПМИ =Прикладная математика и информатика (ПМИ)
   - ПИ = Программная инженерия (ПИ)
   - ПАД = Прикладной анализ данных (ПАД)
   - КБ = Компьютерная безопасность (КБ)
   - ЭАД = Экономика и анализ данных (ЭАД)
   - СПО = Среднее профессиональное образование (СПО)
   - ЕГЭ = Единый государственный экзамен (ЕГЭ)
   - БВИ = Без вступительных испытаний (БВИ)
   - ЕПГУ = Единый портал государственных услуг (ЕПГУ)
   - ГТО = Готов к труду и обороне (ГТО)
2. ЗАМЕНЫ: Если в вопросе есть слово "предметы", заменяй его на "вступительные испытания" или "учебные дисциплины".
3. ФОКУС: Не используй общие фразы про "компетенции", если их нет в оригинале.
4. ФОРМАТ: Пиши ТОЛЬКО текст запроса на русском языке. Без кавычек и лишних фраз.

ПРИМЕРЫ:
- Вопрос: "Как поступить с БВИ на ЭАД?" -> Запрос: "Правила приема без вступительных испытаний (БВИ) на программу Экономика и анализ данных (ЭАД) НИУ ВШЭ"
- Вопрос: "Баллы ЕГЭ для СПО" -> Запрос: "Минимальные баллы Единого государственного экзамена (ЕГЭ) для абитуриентов со средним профессиональным образованием (СПО)"
- Вопрос: "Льготы за ГТО на ПМИ" -> Запрос: "Дополнительные баллы за золотой значок Готов к труду и обороне (ГТО) при поступлении на Прикладную математику и информатику (ПМИ)"

ИСХОДНЫЙ ВОПРОС: {q}

УЛУЧШЕННЫЙ ПОИСКОВЫЙ ЗАПРОС:
"""

    try:
        # Добавлена температура 0 для стабильности rewrite
        rewrite_res = ollama.generate(model='qwen2.5:7b', prompt=rewrite_prompt, options={'temperature': 0})
        improved_q = rewrite_res['response'].strip()
        q =improved_q
    except Exception as e:
        print(f"Ошибка на индексе {index}: {e}")

      
    q_emb = model_embed.encode(q, normalize_embeddings=True).tolist()
    search_res = collection.query(query_embeddings=[q_emb], n_results=3)
    all_texts = search_res['documents'][0]
    
   
    hit, rr = 0, 0
    for rank, doc_text in enumerate(all_texts, 1):
        if any(word.lower() in doc_text.lower() for word in expected_ans.split()[:3]):
            hit, rr = 1, 1/rank
            break
    hit_at_3.append(hit)
    reciprocal_ranks.append(rr)

    context = ""
    for text in all_texts:
        context += text + "\n---\n"
        main_meta = search_res['metadatas'][0][0]

    prompt = f"""
    SYSTEM: You are a professional assistant for HSE University (НИУ ВШЭ). 
    Your tone must be official, formal, and strictly professional. 
    Language: Russian ONLY.

    ИНСТРУКЦИЯ:
    Ты — ведущий консультант приемной комиссии НИУ ВШЭ. Дай краткий и точный ответ на основе предоставленного КОНТЕКСТА.

    ПРАВИЛА:
    1. ФОРМАТ: Не используй технические заголовки типа "ПРЯМОЙ ОТВЕТ:" или "ШАБЛОН:". Просто пиши текст ответа.
    2. ЛОГИКА: Если в контексте ЕСТЬ ответ на вопрос, пиши ТОЛЬКО его. Не добавляй фразы о том, что "информация не зафиксирована" для других программ.
    3. ТОЧНОСТЬ: Пиши только про ту программу, о которой спросил пользователь. Не приводи данные по "Информатике и выч. технике", если вопрос про "ПМИ".
    4. ОТКАЗ: Используй фразу "К сожалению, в предоставленных документах нет информации по данному вопросу" ТОЛЬКО если в контексте вообще нет данных по теме.
    5. ОФОРМЛЕНИЕ: Ключевые цифры и названия программ выделяй **жирным шрифтом**.

    ПОРЯДОК ВЫВОДА:
    - Сформулированный ответ на вопрос.
    - Источник (название документа, ссылка и страница) в отдельной строке.

    КОНТЕКСТ:
    {context}

    ВОПРОС:
    {q}

    ОТВЕТ:
    """

    try:
      response = ollama.generate(model='qwen2.5:7b', prompt=prompt)
    
      bot_answer = response['response'] 
    except Exception as e:
        print(f"Ошибка на индексе {index}: {e}")
        bot_answer = "Ошибка: " + str(e)

    metrics = get_llm_score(q, row['expected_answer'], bot_answer)
    results.append({
        "Номер": index + 1,
        'Вопрос': not_improved_q,
        "Вопрос_улучшенный": q ,
        "Эталонный ответ": row['expected_answer'],
        "Ответ бота": bot_answer,
        "Hit": hit, 
        "Rank": rr, 
        "Категория": main_meta['category'],
        "Ссылка": main_meta['url'],
        "Страница": main_meta['page'],
        "Источник файла": main_meta['source'],
        "LLM Точность": metrics['accuracy'],
        "LLM Полнота": metrics['completeness'],
        "LLM Пояснение": metrics['explanation'],
    })
    cnt+=1
    # if cnt == 5:
    #   break



output_path = os.path.join(os.getcwd() , "final_report_loc.xlsx")
report_df = pd.DataFrame(results)
report_df.to_excel(output_path, index=False)

print('OK')
print(f"Обработано: {cnt}")
print(f"Hit Rate @ 3: {np.mean(hit_at_3):.2f}")
print(f"MRR @ 3: {np.mean(reciprocal_ranks):.2f}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 85804.05it/s]


OK
Обработано: 49
Hit Rate @ 3: 0.90
MRR @ 3: 0.83
